# Transforming Collections with Streams

In this lesson, you will learn to transform and select collection entries with stream pipelines and explain the resulting values and order.

CSC-239 · Module 10 · Lesson 2 of 4

You have passed lambdas and method references to methods that use a chosen rule. Here, you will supply small rules to process collection entries and build reports. You will trace complete examples, repair a pipeline, and create a report with your own tests.

Use the [Module 10 terminology glossary](terms.md) to revisit terms after their explanations in the lesson.


## Learning Goals

- Build and explain a `map`/`filter`/`toList` pipeline over ordered String entries.
- Trace how stage order affects values while retaining required duplicates and leaving the source unchanged.
- Construct and test a report, including empty-input and no-match cases.


## Why This Matters

Reports often need to clean data, choose relevant entries, and present the result. Keeping these steps distinct makes a rule easier to locate and change. It also makes their order visible: checking text before cleaning it can accept or reject a different entry than checking afterward.

Earlier lessons used loops to build separate result lists and functional values to supply behavior. Stream pipelines connect those ideas. They let a report describe a sequence of small rules while preserving the original collection for another part of the program. Without that separation, reporting code can accidentally change data another feature still needs.

A loop remains useful when explicit, multi-step control is clearer. The goal here is to choose and explain a suitable sequence of operations, then test whether it produces the intended report.


## Check Your Starting Point

Recall how a filtering loop decides which elements to keep and how a set differs from a list when values repeat. Explain what `String::trim` supplies and when a lambda body runs. Distinguish supplying a rule from invoking it with an input.


In [ ]:
Your response:

Filtering loop decision:
List versus set with repeats:
What String::trim supplies:
When a lambda body runs:


<details>
<summary>Show answer</summary>

A filtering loop tests each entry against a condition and adds a retained entry to a separate result list when that condition is true. Equal values can remain as separate list entries. A set uses membership rules that prevent duplicate elements.

`String::trim` supplies a compatible operation: when it is invoked with a String, that String becomes the receiver of a `trim()` call. The call returns text with the relevant surrounding whitespace removed. It does not modify the String in place.

Assigning a lambda supplies a functional value. Its body runs when the operation is invoked, such as through `apply` in the previous lesson. Merely creating or passing the rule is not the same as processing an input. Confusing those actions makes it harder to identify where the actual work happens.

</details>


## Video Demonstration

Follow a name report from its ordered source through cleaning, selection, and the completed result. The demonstration shows how each stage's position determines the value its rule receives.

<video controls preload="metadata" width="960">
  <source src="media/02_transforming_collections_with_streams/demo.mp4" type="video/mp4">
  <track kind="captions" src="media/02_transforming_collections_with_streams/captions.vtt" srclang="en" label="English">
  Your browser does not support embedded video.
</video>

[Read the transforming collections with streams demonstration transcript](media/02_transforming_collections_with_streams/transcript.md).


## Concept

### Describe a computation over collection entries

A campus supply desk keeps requested item names in an ArrayList. Some entries include extra spaces. The desk needs a cleaned report that selects entries by a rule while keeping the original requests available. We will first use small item lists to see each operation, then apply the same ideas to a name report.

Earlier lessons built separate result lists with loops and passed behavior as lambdas or method references. A **stream pipeline** provides another way to describe a sequence of operations over elements. The collection supplies the elements; the pipeline describes how to process them and produce a result.

```java
List<String> result = source.stream().toList();
```

This statement comes from the complete two-item example below. `source` is its ArrayList containing `map` followed by `kit`. The call to `stream()` starts a collection-processing stream over those entries. It does not create another ArrayList or remove items from the source. The final `toList()` call produces the result we can read afterward.

The word stream also appears in file input and output, where data moves to or from a file. Here, we are processing elements already held in a collection. No file or file operation is involved. The shared name does not make these APIs interchangeable.


### Separate processing stages from the result

An **intermediate operation** adds a processing stage to a pipeline. For example, `map` describes a transformation to apply to processed elements. A **terminal operation** completes the pipeline's computation and produces its final result or effect. Our examples use `toList()` to obtain a result list. Intermediate operations describe work; the terminal operation initiates processing of the pipeline.

**List** is a Java library interface for an ordered sequence that can contain repeated elements. `List<String>` describes a list whose elements are Strings. This uses the interface and generic-type ideas from earlier lessons. `List` is not a Java keyword. Its import makes the library type available by its short name.

The first complete example has no intermediate transformation or selection stage. It calls `stream().toList()` directly, so both supplied items belong in the result. The result variable lets the following enhanced for loop visit those entries and print each one. Constructing a result and displaying it are separate operations.

After running the example, the output should be `map` followed by `kit`, on separate lines. These are the source's two entries in their original order. Next, we will insert a transformation between the source and the terminal operation.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("map");
source.add("kit");
List<String> result = source.stream().toList();
for (String item : result) {
    System.out.println(item);
}

The complete program produces:

```text
map
kit
```

The terminal operation produces `result` before the enhanced for loop reads it. There is no transformation or selection stage here, so both original entries reach the result in source order. The next example keeps this structure and adds cleaning between `stream()` and `toList()`.


### Transform each processed entry

**Mapping elements** means applying a function to each processed input to obtain an output value. The supply desk can map padded item names to cleaned names. In the next complete example, the source contains `"  map  "` and `" kit "`.

```java
List<String> cleaned = source.stream()
    .map(String::trim)
    .toList();
```

The first line identifies the source and declares where the final list will be stored. The following dot continues that same expression on another line. `map` receives `String::trim`, the method reference taught in the previous lesson. Each incoming String becomes the receiver of a `trim()` call. Its returned String becomes the value passed to the next stage.

The last line collects those returned values. The semicolon ends the complete declaration and assignment; the line breaks above it make the pipeline easier to read. In this example, the padded map entry yields `map`, and the padded kit entry yields `kit`.

The printing loop adds square brackets around each result so the text boundaries are visible. It displays `[map]` and `[kit]`. The final statement reads the first original entry and displays `Source: [  map  ]`. Its spaces remain. Using the returned trimmed String does not replace the entry held in the source ArrayList, and it does not modify a String in place.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("  map  ");
source.add(" kit ");
List<String> cleaned = source.stream()
    .map(String::trim)
    .toList();
for (String item : cleaned) {
    System.out.println("[" + item + "]");
}
System.out.println("Source: [" + source.get(0) + "]");

The brackets make leading and trailing spaces visible:

```text
[map]
[kit]
Source: [  map  ]
```

The first two lines read the cleaned result. The third reads `source.get(0)`, so it shows the original two spaces on each side of `map`. The two reads concern different lists. Assigning the pipeline result to `cleaned` has not assigned new values to the entries of `source`.

Mapping answers what value each processed entry becomes. Next, filtering answers whether that value belongs in the report.


### Keep entries that satisfy a rule

Cleaning solves the spacing problem. The desk can then choose which cleaned entries belong in a report. **Filtering elements** means evaluating a Boolean rule for each processed entry and allowing that entry to continue only when the result is true.

```java
List<String> selected = source.stream()
    .map(String::trim)
    .filter(item -> item.length() == 3)
    .toList();
```

This excerpt belongs to the next complete example, whose source entries are `"  map  "`, `"a"`, `"kit"`, and `"map"`, in that order. Mapping comes first. The filter therefore receives the cleaned value, not the original padded text.

The lambda parameter `item` names the String being tested. `item.length()` obtains its length, and `== 3` asks whether that count equals three. These examples use ordinary English letters, so the count matches their number of letters. The Boolean result controls selection; it does not become an element in the result list.

The first entry becomes `map`, whose length is three, so it continues. The next remains `a`, whose length is one, so it is rejected. `kit` passes with length three. The last `map` also passes. The result is therefore `map`, `kit`, and `map`, followed by a reported selected count of three. Both map entries remain because each independently satisfies the rule. Filtering does not remove duplicates merely because their text matches.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> source = new ArrayList<String>();
source.add("  map  ");
source.add("a");
source.add("kit");
source.add("map");
List<String> selected = source.stream()
    .map(String::trim)
    .filter(item -> item.length() == 3)
    .toList();
for (String item : selected) {
    System.out.println(item);
}
System.out.println("Selected: " + selected.size());

The complete report prints:

```text
map
kit
map
Selected: 3
```

A logical trace connects each original entry to the value tested by the filter:

| Source value | After trim | Length equals three? | Result entry |
| --- | --- | --- | --- |
| `"  map  "` | `"map"` | true | `"map"` |
| `"a"` | `"a"` | false | none |
| `"kit"` | `"kit"` | true | `"kit"` |
| `"map"` | `"map"` | true | `"map"` |

The table describes each stage's effect; it does not imply that the pipeline stores a separate collection for every column. The rejected `a` does not reach the result. Both matching `map` entries do, and `kit` keeps its position between them.

An operation can appear more than once in a pipeline. For example, one mapping can clean text before selection and another can format retained text afterward. Each rule receives the value available at its own position. A mapping after a filter receives only the entries that passed that filter.


<details class="animation-panel" open>
<summary>Trace values through mapping, filtering and collection — show or hide animation</summary>
<p><img src="media/02_transforming_collections_with_streams/map-filter-ordered-collection.gif" alt="Four source entries, padded map, a, kit, and map, pass through cleaning and a rule that keeps text with length three. Both map entries and kit remain in order; a is excluded. The completed result is printed while the original source entries remain unchanged." width="960" style="max-width:100%;height:auto;"></p>
</details>

Each view follows one source entry through the same terminal computation. Mapping supplies cleaned text to the length rule. The repeated map entry passes again and keeps its relative order. The later loop prints the completed result; the original source remains unchanged. This silent loop lasts 22 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/02_transforming_collections_with_streams/map-filter-ordered-collection_still.png).


### Know what the result promises

**Encounter order** is the order in which a source defines its elements for processing. Our ArrayList has a defined element order. These map and filter pipelines keep the relative order of entries that reach the result. Repeated matching entries also remain. An unordered source, such as the HashSet examples from earlier lessons, does not promise this ArrayList ordering.

The `toList()` terminal operation produces an **unmodifiable result list**. You can inspect its entries, size, and order. Attempts to add or remove entries, or replace an entry through that result list, throw `UnsupportedOperationException`. A `List` variable does not by itself promise that edits are supported; the operation that created the particular list matters. The next lesson will exercise this restriction directly. Here, our programs only read the result.

The order of stages also matters because it determines the value each rule receives. The following comparison uses a new supplies list containing `"  ink  "`, `"pen"`, `"a"`, and `"ink"`. Both reports select entries with length three, but one trims before testing and the other tests before trimming.

```java
List<String> trimFirst = supplies.stream()
    .map(String::trim)
    .filter(item -> item.length() == 3)
    .toList();
```

This first pipeline cleans the padded ink entry before measuring it. Its cleaned length is three, so it remains.

```java
List<String> filterFirst = supplies.stream()
    .filter(item -> item.length() == 3)
    .map(String::trim)
    .toList();
```

This second pipeline tests the original entry's length. Two spaces, three letters, and two more spaces make seven, so that entry is rejected before mapping can clean it. The already-unpadded `pen` and `ink` pass in both reports; `a` fails in both. Each pipeline requests its own stream from the same unchanged source. The complete example below prints both results and then the original first entry so you can connect stage order to the observed difference.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> supplies = new ArrayList<String>();
supplies.add("  ink  ");
supplies.add("pen");
supplies.add("a");
supplies.add("ink");
List<String> trimFirst = supplies.stream()
    .map(String::trim)
    .filter(item -> item.length() == 3)
    .toList();
List<String> filterFirst = supplies.stream()
    .filter(item -> item.length() == 3)
    .map(String::trim)
    .toList();
System.out.println("Trim then filter:");
for (String item : trimFirst) {
    System.out.println(item);
}
System.out.println("Filter then trim:");
for (String item : filterFirst) {
    System.out.println(item);
}
System.out.println("Source first: [" + supplies.get(0) + "]");


The two reports differ even though they start from the same unchanged list:

```text
Trim then filter:
ink
pen
ink
Filter then trim:
pen
ink
Source first: [  ink  ]
```

In the first report, the padded entry becomes `ink` before its length is checked. In the second, its original length of seven fails the check, so that entry never reaches the mapping stage. Mapping later cannot bring a rejected entry back. The already-clean `pen` and `ink` pass in both reports; `a` fails in both. The final line confirms that the padded source entry still exists with its original spaces.

The same source-to-result distinction matters at a boundary. An empty source supplies no entries. A nonempty source can also produce an empty result if every entry fails the filter. Both result sizes are zero, but the reasons differ. Keep the source unchanged while a pipeline processes it, and use the completed result as the report.


<details class="animation-panel" open>
<summary>Compare what each stage receives — show or hide animation</summary>
<p><img src="media/02_transforming_collections_with_streams/stage-order-changes-input.gif" alt="Supplies contains padded ink, pen, a, ink in that order. First terminal computation trims before length==3: padded ink becomes ink length3; results are ink, pen, ink. Second terminal computation tests original lengths first: padded ink length7 is rejected; results are pen, ink. Print both result lists in order and Source first: [  ink  ]; the original source entry retains its padding." width="960" style="max-width:100%;height:auto;"></p>
</details>

The first pipeline cleans each input before checking its length. The second checks each original length first, so the padded ink entry fails before it can be trimmed. Each pipeline finishes its own list before the later printing phase. Neither pipeline replaces source entries. This silent loop lasts 22 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/02_transforming_collections_with_streams/stage-order-changes-input_still.png).


## Worked Example

### Build a cleaned name report

A registration report starts from four entries: `"  Maya  "`, `"Li"`, `" Luis "`, and empty text `""`. The report should keep names with at least four characters after surrounding spaces are removed. It should preserve the remaining entries' source order, print each selected name, and report how many entries remain.

**Supply the ordered input.** The program imports `ArrayList` for the source and `List` for the result type. The four `add` calls establish the source order. Empty text is a real String value with length zero; it is different from a missing value.

**Clean before testing length.** These connected lines come from the complete program below:

```java
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
```

The first line requests a stream over `names` and declares `selected`, which will hold the final list. The method reference supplied to `map` returns cleaned text for each processed name. The next line supplies a lambda to `filter`: its parameter is the cleaned String, and `>= 4` is true when that String has at least four characters. The terminal `toList()` produces the list of retained values. Its semicolon ends the whole declaration and assignment.

**Read the completed report.** The enhanced for loop visits `selected` after the pipeline completes. Each `println` displays one retained name. The final statement reads `selected.size()` and prints the count with the label `Selected: `. It measures the report, not the original input list.

The following complete example puts the input, pipeline, and reporting steps together.


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Maya  ");
names.add("Li");
names.add(" Luis ");
names.add("");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());


Expected output:

```text
Maya
Luis
Selected: 2
```

Cleaning produces `Maya`, `Li`, `Luis`, and empty text. Their lengths are four, two, four, and zero. The filter keeps Maya and Luis, in that order, because both meet the minimum of four. The loop therefore prints two names, and `selected.size()` supplies the number two for the last line.

The source still contains its four original entries, including their original spaces. This program reads a report; it does not replace or remove source entries. The next tasks vary the inputs and rules so you can check those distinctions yourself.


## Guided Practice

Work through the prediction and source comparison before completing, modifying, and repairing pipelines. Keep your own reasoning in the response cells before opening the answers. The empty Java cells are places to construct complete programs.


### Predict the selected names

Before running the complete program below, predict every printed line, including the count. Track both occurrences of Iris separately. For each source entry, record the cleaned value and whether it meets the length rule. State the result order and explain which operation produces the result list.


In [ ]:
Your response:

Predicted printed lines:
Each cleaned value and true/false decision:
Result order and repeats:
Operation producing the list:


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("Bo");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());


Run the complete prediction program. Keep your original prediction and record every actual output line. Explain any correction.

Write one trace row for each source entry: original text, result of `trim`, result of the length condition, and retained result if any. Identify the source, intermediate operations, and terminal operation. Explain the different jobs of the method reference and the Boolean-result lambda.

Explain why both Iris entries remain and why Owen stays between them. Would a HashSet source promise this same order? Use the source's ordering rule in your explanation.


In [ ]:
Your response:

Actual lines and corrections:
Trace: source / after trim / condition / retained result:
Source, intermediate operations, and terminal operation:
Method reference versus lambda:
Repeated entries and order; HashSet comparison:


<details>
<summary>Show answer</summary>

The cleaned values are Iris, Bo, Owen and Iris, in source order. The length rule keeps Iris and Owen because each has four characters. Bo has two characters and is left out. The later Iris also meets the rule, so both Iris entries remain. The list therefore prints Iris, Owen, Iris and then Selected: 3. map supplies cleaned values to filter; the filter rule returns true or false for each value it receives. toList is the terminal operation that produces selected. The printing loop reads that completed result. stream supplies the source; map and filter are intermediate operations. String::trim supplies the mapping operation, while the lambda supplies the length condition. The result is a List of String values in the surviving entries’ encounter order. These stages neither sort the entries nor remove equal entries automatically. The list source provides the order; a HashSet source would not provide the same promised iteration order.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("Bo");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Iris
Owen
Iris
Selected: 3
```

Common error: Counting surrounding spaces after trim has already removed them. Keeping only one occurrence of an equal name. Assuming the result will be sorted.

</details>


### Compare the source with the result

The complete program below adds two lines that inspect the source after producing the report. Predict its entire output before running, including both source-check lines. Use the brackets to show the original spaces exactly.


In [ ]:
Your response:

Predicted report lines:
Predicted source-check lines, including exact spaces:


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("Bo");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
System.out.println("Source first: [" + names.get(0) + "]");
System.out.println("Source entries: " + names.size());


Run the complete program and record its actual output. Compare the two source-check lines with your prediction. Explain whether `map` changed the original first entry and whether `filter` removed Bo from the source. Keep the result list and the source list distinct in your explanation.


In [ ]:
Your response:

Actual output and comparison:
Did map replace the first source entry?
Did filter remove Bo from the source?


### Reorder the original entries

Change only the four `add` statements in the program above so the source entries are `" Owen "`, `"Iris"`, `"Bo"`, and `"  Iris  "`, in that order. Before editing and running, predict every new output line, including the first source entry with its exact spaces and the source count.


In [ ]:
Your response:

Reordered-input prediction, including source checks:


Edit and run the complete source-comparison program for that reordered input. Record the actual output and explain the new result order and the original first entry. Compare your prediction with the result, then restore the original input order and rerun. Record the restored output as a check.


In [ ]:
Your response:

Actual reordered output:
Result order and first source entry:
Corrections:
Restored original output:


<details>
<summary>Show answer</summary>

The cleaned values are Iris, Bo, Owen and Iris, in source order. The length rule keeps Iris and Owen because each has four characters. Bo has two characters and is left out. The later Iris also meets the rule, so both Iris entries remain. The list therefore prints Iris, Owen, Iris and then Selected: 3. map supplies cleaned values to filter; the filter rule returns true or false for each value it receives. toList is the terminal operation that produces selected. The printing loop reads that completed result. The extra lines read the original names list after the pipeline. Its first entry still contains two surrounding spaces on each side of Iris, so brackets reveal `Source first: [  Iris  ]`. Its size is still four, including Bo. This pipeline produced selected without replacing or removing the source entries. The reordered comparison places Owen before both retained Iris entries; its result follows that changed list order. The original first value in that comparison remains the padded Owen text.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("Bo");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
System.out.println("Source first: [" + names.get(0) + "]");
System.out.println("Source entries: " + names.size());
```

Expected output:

```text
Iris
Owen
Iris
Selected: 3
Source first: [  Iris  ]
Source entries: 4
```

Common error: Assuming a cleaned result replaces the padded source entry. Assuming a rejected result entry disappears from the source list. Treating a duplicate name as a reason to discard an entry.

**Check case 2.** The list now encounters padded Owen first, then Iris, Bo and padded Iris. After cleaning and filtering, Owen precedes both Iris entries. Bo remains in the original four-entry source, even though it is absent from selected. Brackets show the original spaces around Owen.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add(" Owen ");
names.add("Iris");
names.add("Bo");
names.add("  Iris  ");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
System.out.println("Source first: [" + names.get(0) + "]");
System.out.println("Source entries: " + names.size());
```

Expected output:

```text
Owen
Iris
Iris
Selected: 3
Source first: [ Owen ]
Source entries: 4
```

The animation returns to the first source order: padded Iris, Bo, padded Owen, and Iris. Follow that result alongside its unchanged source.

<details class="animation-panel" open>
<summary>Read the result and the original source separately — show or hide animation</summary>
<p><img src="media/02_transforming_collections_with_streams/separate-source-and-result.gif" alt="The source list contains padded Iris, Bo, padded Owen, and Iris. Cleaning and filtering produce Iris, Owen, and Iris in that order. The source still has four entries, including Bo and the original padding; the result has three entries." width="960" style="max-width:100%;height:auto;"></p>
</details>

The completed result has three entries: Iris, Owen and Iris. The original names list still has four entries, including Bo and the spaces around the first Iris. The printing loop reads the result; the later get and size calls read the original source. This silent loop lasts 16 seconds. Hiding it removes the visible motion. [View the final state as a still image](media/02_transforming_collections_with_streams/separate-source-and-result_still.png).

</details>


### Complete the connected operations

Replace `SOURCE`, `TRANSFORM`, `CHOOSE`, and `FINISH` in the displayed program. Use `stream`, `map`, `filter`, and `toList` once each in the positions that supply the source, clean each word, select words of length four, and produce the result list. Keep the operation order shown.

Before coding, record each proposed replacement and the role it plays. Explain why the semicolon follows the completed expression. The required output is `tree`, `leaf`, and `Selected: 2` on separate lines.

This incomplete sample is for repair. Put the full completed program in the Java work cell and run it.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> words = new ArrayList<String>();
words.add("  tree  ");
words.add("ox");
words.add(" leaf ");
List<String> selected = words.SOURCE()
    .TRANSFORM(String::trim)
    .CHOOSE(word -> word.length() == 4)
    .FINISH();
for (String word : selected) {
    System.out.println(word);
}
System.out.println("Selected: " + selected.size());
```


In [ ]:
Your response:

Four replacements and their roles:
Why the semicolon follows the complete expression:


Record the actual output and compare it with the target. Explain how your four replacements connect the source, cleaning rule, selection rule, and completed result. Describe any correction you made after running.


In [ ]:
Your response:

Actual output:
How the four operations connect:
Corrections:


<details>
<summary>Show answer</summary>

SOURCE is stream, TRANSFORM is map, CHOOSE is filter and FINISH is toList. The cleaned words are tree, ox and leaf. The condition is true for tree and leaf and false for ox. toList creates the result list in that surviving order. The semicolon ends the entire assignment expression after toList; the line breaks only make its operations easier to read. The program includes both imports, all word entries, the pipeline and the printing loop.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> words = new ArrayList<String>();
words.add("  tree  ");
words.add("ox");
words.add(" leaf ");
List<String> selected = words.stream()
    .map(String::trim)
    .filter(word -> word.length() == 4)
    .toList();
for (String word : selected) {
    System.out.println(word);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
tree
leaf
Selected: 2
```

Common error: Ending the assignment before its terminal operation. Placing the selection before cleaning even though the rule concerns cleaned length. Leaving a placeholder in executable code.

</details>


### Change the selection boundary

The starter is the complete prediction program. Change only its filter so it keeps cleaned names with at least two characters. Keep the mapping and terminal operation unchanged.

Before editing, predict which entries will remain and their order, including the printed count. Explain how the new boundary applies to the cleaned value. Then make this change in the complete starter and run it.


In [ ]:
Your response:

Predicted two-character-boundary output:
How the cleaned-length boundary applies:


In [ ]:
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("Bo");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());


Record every actual output line after lowering the minimum length to two. Explain which entry the changed boundary now retains and how the other entries keep their relative order.


In [ ]:
Your response:

Actual two-character-boundary output:
Changed entry and retained order:


### Check a value below the new boundary

Keep the minimum length of two. Next, change only the source value `"Bo"` to `"A"`. Predict every output line and explain how the rule applies to that cleaned input. Then edit and run the same complete program above.


In [ ]:
Your response:

Predicted A-test output:
Why the changed entry passes or fails:


Record the actual A-test output. Compare it with the previous two-character case and explain why the changed entry passes or fails. Restore Bo while keeping the minimum length of two, rerun, and record that restored result.


In [ ]:
Your response:

Actual A-test output and comparison:
Boundary explanation:
Restored Bo output with minimum two:


<details>
<summary>Show answer</summary>

Changing the comparison from >= 4 to >= 2 keeps all four cleaned names: Iris, Bo, Owen and Iris. Their relative source order and both Iris entries remain. After replacing Bo with A, the cleaned one-character value fails the same condition, so the result returns to Iris, Owen and Iris with size three. Only the rule boundary and then one input change; the cleaning and result-building stages retain their jobs.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("Bo");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 2)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Iris
Bo
Owen
Iris
Selected: 4
```

Common error: Changing the raw source text to force an entry to pass. Changing trim or toList when only the selection rule needs adjustment. Dropping one of the equal Iris entries.

**Additional test: `Minimum length two; replace Bo with A`.** A has one character after cleaning and fails the two-character minimum. Iris, Owen and the later Iris remain in their source order.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("A");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 2)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Iris
Owen
Iris
Selected: 3
```

</details>


### Repair a length check that runs too early

The displayed program runs, but its goal is to keep names whose cleaned length is at least four. Predict the faulty output before editing. Explain the exact value and length received by the condition for the padded short name `" Bo "`.

Plan a repair to the operation order so cleaning happens before selection. Keep all source entries and print statements. After recording your diagnosis, put the complete repaired program in the Java work cell and run it.

This sample is intentionally faulty:

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add(" Bo ");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .filter(name -> name.length() >= 4)
    .map(String::trim)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
```


In [ ]:
Your response:

Predicted faulty lines:
Value and length seen by the faulty condition:
Planned operation-order repair:


Record the actual repaired output. Explain the value the length condition now receives for the padded short name and why that differs from the faulty program. Identify the two stages you reordered.


In [ ]:
Your response:

Actual repaired output:
Input to the condition before and after repair:
Reordered stages:


### Check whitespace-only input

In your repaired program, replace `" Bo "` with four spaces, `"    "`. Keep the repaired operation order and all other inputs and print statements. Predict every printed line and explain what the condition will receive after cleaning. Then edit and run your complete repaired program.


In [ ]:
Your response:

Predicted four-space test lines:
Cleaned value and selection decision:


Record the actual four-space test output and compare it with your prediction. Explain why both the padded short name and whitespace-only text expose the need to check cleaned text. Restore `" Bo "`, keep the repair, rerun, and record the restored result.


In [ ]:
Your response:

Actual four-space test output and comparison:
Why both cases check the repair:
Restored padded-Bo output:


<details>
<summary>Show answer</summary>

The faulty condition sees `" Bo "` before trim. Its two letters and two spaces make a length of four, so it passes too soon; mapping later changes it to Bo without repeating the filter. The faulty result therefore includes Bo. Moving map(String::trim) before filter makes the condition receive Bo with length two, which is rejected. The repaired result is Iris, Owen, Iris and Selected: 3. Four spaces become empty text after trim and are also rejected by the repaired condition. The program must clean each value before checking the length of that cleaned value.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add(" Bo ");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Iris
Owen
Iris
Selected: 3
```

Common error: Keeping the faulty order and changing the expected result instead. Assuming trimming after filter causes filter to reconsider the entry. Changing the supplied input to hide the order error.

**Additional test: `Four spaces replace the padded short name`.** The repaired map turns four spaces into empty text before the length comparison. That result fails the minimum of four, so no blank entry is printed. The remaining three names keep their relative order.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> names = new ArrayList<String>();
names.add("  Iris  ");
names.add("    ");
names.add(" Owen ");
names.add("Iris");
List<String> selected = names.stream()
    .map(String::trim)
    .filter(name -> name.length() >= 4)
    .toList();
for (String name : selected) {
    System.out.println(name);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Iris
Owen
Iris
Selected: 3
```

</details>


## Independent Practice

### Build the request-packing report

A supply desk needs one packing line for each request for a pen. Separate equal requests must remain separate report entries.

Start an ArrayList named `requests` with these five String values, in order: `" pen "`, `"kit"`, `"  map  "`, `"pen"`, and `" "`. Build a pipeline that trims each item, retains cleaned items equal to `pen`, then maps each retained item to `"Pack: "` followed by the item. Use `toList` to produce a `List<String>` named `selected`.

Print every selected entry in order and then `"Selected: "` followed by its size. The baseline should print two `Pack: pen` lines and `Selected: 2`. Keep repeats.

Before coding, describe the value each rule receives and the result it produces. Explain why cleaning precedes equality testing and which input the final mapping receives. Then write your own complete program in the Java work cell, including both imports, all input setup, the pipeline, and printing code, and run it.


In [ ]:
Your response:

Input and result at each stage:
Why cleaning precedes equality:
Input to the final mapping:
Expected baseline:


Record the actual baseline output. Trace the cleaned inputs and equality decisions, then identify the input and result of the final mapping. Explain why both matching entries remain and how the printed count relates to the result list.


In [ ]:
Your response:

Actual baseline output:
Cleaned inputs and equality decisions:
Final mapping input and result:
Repeated matches and count:


### Check empty and repeated-request cases

Keep the pipeline and output statements unchanged while testing these three separate inputs:

1. Replace the original five entries with `"kit"`, `"  map  "`, and `" "`.
2. Use an empty `requests` list.
3. Restore the original five entries and append another `"  pen  "`.

Before running these variants, predict the exact printed lines for each case. Label every prediction, including the original five-entry case that you will restore after testing.


In [ ]:
Your response:

No-match prediction:
Empty-source prediction:
Additional-pen prediction:
Restored baseline prediction:


Edit and run the complete report with each planned input. Record the actual output for the no-match case, empty-source case, and additional-pen case. Compare each result with your prediction and explain any correction.

Distinguish an empty source from a nonempty source with no matches. Explain why the extra padded match remains as another entry. Finally restore and rerun the original five-entry case, and record its output.


In [ ]:
Your response:

Actual no-match output and reason:
Actual empty-source output and reason:
Actual additional-pen output and reason:
Corrections:
Restored baseline output:


<details>
<summary>Show answer</summary>

The first map produces pen, kit, map, pen and empty text. The equality filter receives those cleaned values, so both pen entries pass. The second map receives each retained pen and returns Pack: pen, including the space after the colon. toList produces a two-entry list in the original relative order. The loop prints both entries and the final line reports Selected: 2. Cleaning must happen before equality testing so the padded pen is compared as pen. The separate second map formats only retained items. All imports and request setup are included. The no-match case has three source entries, but none becomes pen. The empty-input case has no source entries to process. Both produce an empty result, so only Selected: 0 is printed. Adding another padded pen to the original input creates a third matching entry after cleaning; all three remain. Restoring the original five entries verifies the two-entry baseline again.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> requests = new ArrayList<String>();
requests.add(" pen ");
requests.add("kit");
requests.add("  map  ");
requests.add("pen");
requests.add(" ");
List<String> selected = requests.stream()
    .map(String::trim)
    .filter(item -> item.equals("pen"))
    .map(item -> "Pack: " + item)
    .toList();
for (String item : selected) {
    System.out.println(item);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Pack: pen
Pack: pen
Selected: 2
```

Common error: Comparing a padded item with pen before trimming. Formatting every request before testing equality with pen. Treating equal requests as one request.

**Additional test: No matching items: kit, padded map and one space.** Cleaning produces kit, map and empty text. None equals pen, so the result is empty even though the source has three entries. The printing loop has no result entry to visit.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> requests = new ArrayList<String>();
requests.add("kit");
requests.add("  map  ");
requests.add(" ");
List<String> selected = requests.stream()
    .map(String::trim)
    .filter(item -> item.equals("pen"))
    .map(item -> "Pack: " + item)
    .toList();
for (String item : selected) {
    System.out.println(item);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Selected: 0
```

**Additional test: Empty input list.** The source has zero entries. The terminal operation produces an empty result list, and the final size line still runs. This differs from rejecting entries that were present.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> requests = new ArrayList<String>();
List<String> selected = requests.stream()
    .map(String::trim)
    .filter(item -> item.equals("pen"))
    .map(item -> "Pack: " + item)
    .toList();
for (String item : selected) {
    System.out.println(item);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Selected: 0
```

**Additional test: Original inputs plus an additional padded pen.** The extra padded pen becomes pen before comparison and then Pack: pen after the second map. It is retained after the earlier two matches, so three matching entries are printed.

```java
import java.util.ArrayList;
import java.util.List;
ArrayList<String> requests = new ArrayList<String>();
requests.add(" pen ");
requests.add("kit");
requests.add("  map  ");
requests.add("pen");
requests.add(" ");
requests.add("  pen  ");
List<String> selected = requests.stream()
    .map(String::trim)
    .filter(item -> item.equals("pen"))
    .map(item -> "Pack: " + item)
    .toList();
for (String item : selected) {
    System.out.println(item);
}
System.out.println("Selected: " + selected.size());
```

Expected output:

```text
Pack: pen
Pack: pen
Pack: pen
Selected: 3
```

</details>


## Summary

A stream pipeline connects a source with operations that describe a computation. Mapping produces values; filtering decides which values continue; a terminal operation produces the final result or effect. Here, `toList` returns an unmodifiable list that the program reads afterward.

Stage order determines what each rule receives. These pipelines preserve the surviving entries' relative order from an ArrayList, including repeated matching entries. They produce a result without replacing or removing entries in that source.

Close the answers and trace a padded entry, a rejected entry, and a repeated matching entry from the small item example. For each, state its value after cleaning, whether it passes the length-three rule, and whether it reaches the result. Explain which operation produces that result list.


In [ ]:
Your response:

Padded entry trace:
Rejected entry trace:
Repeated matching entry trace:
Operation producing the list:


<details>
<summary>Show answer</summary>

In the small item example, `"  map  "` becomes `"map"`, passes the length-three rule, and reaches the result. `"a"` stays `"a"`, fails the rule, and does not reach the result. The later `"map"` also passes and remains as a separate entry. The retained `kit` stays between the two map entries because the source is ordered.

`map` supplies cleaned values, `filter` controls which continue, and `toList` is the terminal operation that produces the list. The printing loop then reads that completed list. Counting original spaces after cleaning, removing equal matches as though the result were a set, or assuming an alphabetical sort would each give the wrong trace.

</details>


## Reflection

Describe a report in your field that needs both cleaning and selection. Give one input for which swapping those two stages would change the result. Explain whether duplicate values should remain and which source order matters.


In [ ]:
Your response:

Report and its purpose:
Input affected by stage order:
Duplicate policy:
Required source order:


The next lesson distinguishes a pipeline's description from its execution. It will also examine how to reuse results and create editable copies when later work needs to change a list.


## Supplemental Reading

- [Java Stream operations](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/Stream.html) defines map, filter, and toList.
- [Stream pipelines and encounter order](https://docs.oracle.com/en/java/javase/21/docs/api/java.base/java/util/stream/package-summary.html) explains how sources and operations determine a computation.
